# Cross validation


In [8]:
#Contact us on aboulesnane@univ-constantine3.dz

Converting annotations...

Conversion finished.
Images: 634
Labels: 634

Preparing Fold 1
Fold 1 ready.

Preparing Fold 2
Fold 2 ready.

Preparing Fold 3
Fold 3 ready.

Preparing Fold 4
Fold 4 ready.

Preparing Fold 5
Fold 5 ready.

TRAINING FOLD 1
New https://pypi.org/project/ultralytics/8.4.163 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/fold1.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv

In [9]:
# ── START: old vs new CV comparison ──
OLD = {"map50":     [0.9473, 0.9665, 0.9545, 0.9430, 0.9484],
       "map":       [0.5613, 0.5484, 0.5541, 0.5469, 0.5413],
       "precision": [0.9197, 0.9479, 0.9308, 0.9179, 0.9254],
       "recall":    [0.9270, 0.9529, 0.9340, 0.9195, 0.9344]}
from scipy.stats import wilcoxon

print(f"{'Fold':<6}" + "".join(f"{k:>22}" for k in OLD))
for i in range(5):
    print(f"{i+1:<6}" + "".join(f"{OLD[k][i]:>9.4f} -> {results[k][i]:.4f}" for k in OLD))
print("-" * 94)
for k in OLD:
    o, n = np.array(OLD[k]), np.array(results[k])
    d = n - o
    p = wilcoxon(n, o).pvalue if np.any(d != 0) else 1.0
    print(f"{k:<10} old {o.mean():.4f}±{o.std():.4f}   new {n.mean():.4f}±{n.std():.4f}   "
          f"Δ {d.mean():+.4f}   Wilcoxon p={p:.3f}")
# ── END ──

Fold                   map50                   map             precision                recall
1        0.9473 -> 0.9484   0.5613 -> 0.5570   0.9197 -> 0.9269   0.9270 -> 0.9299
2        0.9665 -> 0.9670   0.5484 -> 0.5483   0.9479 -> 0.9424   0.9529 -> 0.9526
3        0.9545 -> 0.9559   0.5541 -> 0.5548   0.9308 -> 0.9307   0.9340 -> 0.9354
4        0.9430 -> 0.9428   0.5469 -> 0.5406   0.9179 -> 0.9248   0.9195 -> 0.9309
5        0.9484 -> 0.9507   0.5413 -> 0.5387   0.9254 -> 0.9322   0.9344 -> 0.9344
----------------------------------------------------------------------------------------------
map50      old 0.9519±0.0082   new 0.9530±0.0082   Δ +0.0010   Wilcoxon p=0.125
map        old 0.5504±0.0068   new 0.5479±0.0073   Δ -0.0025   Wilcoxon p=0.188
precision  old 0.9283±0.0108   new 0.9314±0.0061   Δ +0.0031   Wilcoxon p=0.312
recall     old 0.9336±0.0111   new 0.9366±0.0082   Δ +0.0031   Wilcoxon p=0.312


In [14]:
# ── START: pick the best tooth model by FDI numbering on lesion teeth (same data for all) ──
import os, json
from collections import Counter, defaultdict
from ultralytics import YOLO

CANDIDATES = {f"CV fold {k}": f"/kaggle/working/dentex_cv/fold{k}/weights/best.pt" for k in range(1, 6)}
CANDIDATES["PanorAI r2"] = "/kaggle/working/panorai_runs/nofliplr_r2/weights/best.pt"
REFERENCE = {"OLD Yolo_teeth": "/kaggle/input/models/boulesnane/yolo-teeth/other/default/1/Yolo_teeth.pt"}  # fliplr 0.5, not selectable

IOU_MATCH = 0.30
DX = ("/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/"
      "7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX")
SOURCES = {
    "train": (f"{DX}/training_data/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json",
              f"{DX}/training_data/training_data/quadrant-enumeration-disease/xrays"),
    "val":   (f"{DX}/validation_triple.json",
              f"{DX}/validation_data/validation_data/quadrant_enumeration_disease/xrays"),
}
DISEASE = {0: "Impacted", 1: "Caries", 2: "Periapical", 3: "Deep Caries"}
YAML_NAMES = {i: f"{q}{t}" for i, (q, t) in enumerate((q, t) for q in range(1, 5) for t in range(1, 9))}
MIRROR = {"1": "2", "2": "1", "3": "4", "4": "3"}
STATUSES = ["correct", "off by one", "side swapped", "other wrong", "no tooth box"]

def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

def status_of(g, p):
    if p == g: return "correct"
    if p == MIRROR[g[0]] + g[1]: return "side swapped"
    if p[0] == g[0] and abs(int(p[1]) - int(g[1])) == 1: return "off by one"
    return "other wrong"

lesions = []
for js, img_dir in SOURCES.values():
    d = json.load(open(js)); names = {im["id"]: im["file_name"] for im in d["images"]}
    for a in d["annotations"]:
        x, y, w, h = a["bbox"]
        lesions.append((os.path.join(img_dir, names[a["image_id"]]), [x, y, x + w, y + h],
                        f"{a['category_id_1'] + 1}{a['category_id_2'] + 1}", DISEASE[a["category_id_3"]]))
paths = sorted({l[0] for l in lesions if os.path.exists(l[0])})
print(f"{len(lesions)} lesion teeth in {len(paths)} images | agnostic NMS | IoU match {IOU_MATCH}\n")

def evaluate(model_path):
    mdl = YOLO(model_path); preds = {}
    for p in paths:
        r = mdl.predict(p, imgsz=640, conf=0.25, agnostic_nms=True, verbose=False)[0]
        preds[p] = [(YAML_NAMES[int(c)], b) for b, c in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist())]
    allc, disc = Counter(), defaultdict(Counter)
    for p, gbox, g, dis in lesions:
        if p not in preds: continue
        hits = sorted([(iou(gbox, pb), pf) for pf, pb in preds[p] if iou(gbox, pb) >= IOU_MATCH], reverse=True)
        st = "no tooth box" if not hits else status_of(g, hits[0][1])
        allc[st] += 1; disc[dis][st] += 1
    return allc, disc

pct = lambda c, k: 100 * c[k] / max(sum(c.values()), 1)
scores = {}
print(f"{'Model':<16}" + "".join(f"{s:>14}" for s in STATUSES) + "".join(f"{d:>12}" for d in DISEASE.values()))
print("-" * 134)
for group in (REFERENCE, CANDIDATES):
    for name, path in group.items():
        if not os.path.exists(path):
            print(f"{name:<16} not found: {path}"); continue
        c, dc = evaluate(path)
        if group is CANDIDATES:
            scores[name] = (pct(c, "correct"), -pct(c, "side swapped"), path)
        print(f"{name:<16}" + "".join(f"{pct(c, s):13.1f}%" for s in STATUSES)
              + "".join(f"{pct(dc[d], 'correct'):11.1f}%" for d in DISEASE.values()))
    if group is REFERENCE:
        print("-" * 134)

best = max(scores, key=lambda k: scores[k][:2])       # highest correct, then fewest side swaps
print(f"\nBEST for the pipeline: {best}  ({scores[best][0]:.1f}% correct FDI)  ->  {scores[best][2]}")
# ── END ──

3711 lesion teeth in 724 images | agnostic NMS | IoU match 0.3

Model                  correct    off by one  side swapped   other wrong  no tooth box    Impacted      Caries  Periapical Deep Caries
--------------------------------------------------------------------------------------------------------------------------------------
OLD Yolo_teeth           91.9%          4.4%          1.4%          0.1%          2.2%       96.3%       94.8%       84.4%       78.5%
--------------------------------------------------------------------------------------------------------------------------------------
CV fold 1                93.1%          4.6%          0.1%          0.1%          2.1%       97.0%       95.2%       90.4%       81.6%
CV fold 2                93.3%          4.6%          0.1%          0.1%          2.0%       97.2%       95.4%       93.4%       81.5%
CV fold 3                93.0%          5.3%          0.1%          0.1%          1.5%       97.4%       94.5%       89.2%    